# Understanding a Real IoT Sensor Stream: the Shower Flow Meter (WEUSEDTO)

**Goal.** Describe a real IoT sensor stream in plain terms first, then in more and more detail: what the data looks like, how a time series is checked, when and how the sensor is used, how predictable the signal is, and whether other sensors in the same data set behave the same way.

**Data.** Residential water end-use data from the WEUSEDTO project (licence CC BY 4.0).
Repositories: [`Water-End-Use-Dataset-Tools/WEUSEDTO`](https://github.com/Water-End-Use-Dataset-Tools/WEUSEDTO) and [`AnnaDiMauro/WEUSEDTO-Data`](https://github.com/AnnaDiMauro/WEUSEDTO-Data). Please cite the dataset publication listed in the repository README. The data files are **not** included here.

**How to run.** Download the `WEUSEDTO-Data` repository as a ZIP, unzip it next to this notebook (the folder name starts with `WEUSEDTO-Data-`), then use *Restart & Run All*. Plots go to `figures/`, tables and the profile JSON go to `outputs/`.

**In one paragraph.** The shower meter is idle almost all of the time (98.75% of the time), writes a reading every 1–2 seconds while water flows and about every 5 minutes while idle, and its values sit on a fixed step grid. Flow comes in a few long events, and each reading is very close to the previous one.

**How the notebook is organised (simple to detailed)**

| Part | Sections | Question |
|---|---|---|
| A. Looking at the data | 1–5 | What is in the file, how is a time series checked, what do the events look like, when is water used? |
| B. Looking deeper | 6–9 | How predictable is the signal, how are the values built, how big is the noise, how does time behave? |
| C. Other fixtures and summary | 10–11 | Do the findings hold for the other fixtures? What is the overall picture? |

Every section separates what was **measured** from what is **assumed or inferred**.

## Setup

In [ ]:
import json  # tool to save the profile as a JSON file
from pathlib import Path  # tool to work with file paths

import numpy as np  # tool for number arrays
import pandas as pd  # tool to work with tables
import matplotlib.pyplot as plt  # tool for drawing charts

%matplotlib inline

# ---------------- Settings (edit here) ----------------
OUT_DIR = Path("outputs")  # tables and the profile JSON are saved here
FIG_DIR = Path("figures")  # plots are saved here
TZ = "Europe/Rome"  # ASSUMPTION: the apartment is in Naples, so local time = Europe/Rome
MIN_TS, MAX_TS = 1.5e9, 1.7e9  # ASSUMPTION: valid Unix times are between mid-2017 and late 2023 (data is from 2019-2020)
SENSOR_MIN_ML_S, SENSOR_MAX_ML_S = 1000 / 60, 30000 / 60  # vendor working range of the YF-S201 (1 to 30 L/min) in ml/s; not verified for this unit

candidates = list(Path(".").glob("WEUSEDTO-Data-*"))  # look for the unzipped dataset folder next to this notebook
if not candidates:  # stop with a clear message if it is missing
    raise FileNotFoundError("Unzip the WEUSEDTO-Data repository next to this notebook (folder name starts with 'WEUSEDTO-Data-').")
DATA_DIR = candidates[0] / "Dataset"  # folder with the CSV files

for folder_path in (OUT_DIR, FIG_DIR):  # make sure the output folders exist
    folder_path.mkdir(exist_ok=True)  # create the folder if needed

In [ ]:
def entropy_bits(series):  # entropy in bits per value (higher = harder to guess)
    p = series.value_counts(normalize=True)  # probability of each distinct value
    return float(-(p * np.log2(p)).sum())  # Shannon entropy formula

def cond_entropy(s):  # entropy of the value when the previous value is known
    s = (s - s.min()).astype("int64").reset_index(drop=True)  # shift to non-negative whole numbers
    prev, cur = s.iloc[:-1].reset_index(drop=True), s.iloc[1:].reset_index(drop=True)  # previous and current value
    key = prev * (int(s.max()) + 1) + cur  # one number per (previous, current) pair
    return entropy_bits(key) - entropy_bits(prev)  # H(pair) - H(previous)

def save_and_show(file_name):  # fit the layout, save the figure to FIG_DIR and show it
    plt.tight_layout()  # fit everything in the figure
    plt.savefig(FIG_DIR / file_name, dpi=150)  # save the picture
    plt.show()  # show the chart

# Part A. Looking at the data

## 1. What does the data look like?

Each row is one reading from the sensor: a **time** (Unix seconds, i.e. seconds since 1 January 1970) and a **flow value**. The file has no header row.

In [ ]:
data = pd.read_csv(  # open the file as a table
    DATA_DIR / "feed_Shower.MYD.csv",  # the shower file
    sep=r"\s+",  # columns are separated by spaces
    header=None,  # the file has no header row
    names=["timestamp", "value"],  # our own column names
)  # end of read_csv
values = data["value"]  # short name for the flow column
data.head()  # show the first 5 rows

**Data dictionary.** What each column means and what is documented versus measured.

In [ ]:
data_dictionary = pd.DataFrame([  # one row per column of the file
    {
        "column": "timestamp",  # name used in this notebook
        "meaning": "time of the reading",  # what it is
        "unit": "Unix seconds (UTC)",  # unit
        "type in file": str(data["timestamp"].dtype),  # data type found by pandas
        "range in this file": f"{pd.to_datetime(data['timestamp'].min(), unit='s')} to {pd.to_datetime(data['timestamp'].max(), unit='s')}",  # first and last reading
        "status": "documented",  # where the information comes from
    },
    {
        "column": "value",  # name used in this notebook
        "meaning": "water flow through the shower pipe",  # what it is
        "unit": "ml/s",  # unit given in the dataset publication
        "type in file": str(values.dtype),  # data type found by pandas
        "range in this file": f"{values.min():.0f} to {values.max():.0f}",  # smallest and largest value
        "status": "unit documented; other properties measured below",  # where the information comes from
    },
])  # end of table
display(data_dictionary)  # show the table

**What is known about the sensor** (dataset publication and vendor specification of the sensor type; to be confirmed in the repository README)
- The fixture-level meters are described in the dataset publication as small turbine flow sensors of the **YF-S201** type, read by a microcontroller.
- Vendor specification for that sensor type: about **2.25 ml per pulse** (450 pulses per litre), working range **1–30 L/min**, accuracy about **±10%**, "not a precision sensor".
- **Not documented for this file:** the logging rule (when a reading is written), the counting window, how the data is transmitted, and how the whole-house file is built.

In [ ]:
profile = {  # one dictionary that collects all results
    "context": {  # what we know before looking at the numbers
        "sensor_name": "Shower flow meter",  # which sensor this is
        "sensor_type": "flow",  # kind of sensor
        "source": "WEUSEDTO dataset (CC BY 4.0)",  # where the data comes from
        "value_unit": "ml/s (documented in the dataset publication)",  # unit of the value column
        "nominal_sampling": "1 s (documented, to verify against the data)",  # expected reading interval
        "adc_bits": "unknown",  # sensor resolution, not documented
        "transport": "unknown for this file",  # how data is sent, not documented
    }  # end of context
}  # end of profile

The first plot shows the first readings twice. **Left:** against the reading number (each reading gets the same width). **Right:** against real time. If a sensor sampled once per second, the two pictures would look alike.

In [ ]:
n_show = 1500  # how many readings to draw
first = data.iloc[:n_show]  # the first readings

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))  # two charts side by side
axes[0].plot(range(len(first)), first["value"], linewidth=0.8)  # value against reading number
axes[0].set_xlabel("Reading number")  # label of the horizontal axis
axes[0].set_ylabel("Flow (ml/s)")  # label of the vertical axis
axes[0].set_title("By reading number")  # chart title
axes[1].plot(pd.to_datetime(first["timestamp"], unit="s"), first["value"], marker=".", markersize=2, linewidth=0.5)  # value against real time
axes[1].set_xlabel("Time (UTC)")  # label of the horizontal axis
axes[1].set_title("By real time")  # chart title
axes[1].tick_params(axis="x", rotation=30)  # tilt the dates
save_and_show("plot0_first_readings.png")  # fit, save and show the chart

**What we see.** Most readings are 0, with bursts where water flows. Against real time the readings are not evenly spaced: they are dense during a burst and far apart in between. So this is **not** a "one reading per second" series, which is the first thing to check for any time series (next section).

## 2. First checks of any time series

Before any modelling, four simple questions: **(a)** how big is it and what period does it cover, **(b)** is anything missing, duplicated or out of order, **(c)** how regular is the sampling, **(d)** where are the gaps?

### 2a. Size and period

In [ ]:
profile["data_quality"] = {}  # new section for data quality results
quality = profile["data_quality"]  # short name for that section

quality["row_count"] = len(data)  # number of rows (readings)

start = pd.to_datetime(data["timestamp"].min(), unit="s")  # first reading as a date
end = pd.to_datetime(data["timestamp"].max(), unit="s")  # last reading as a date
quality["start_time"] = str(start)  # save the start date as text
quality["end_time"] = str(end)  # save the end date as text
quality["span_days"] = (end - start).days  # days between first and last reading

print(quality)  # show the results so far

### 2b. Missing values, duplicates, order

In [ ]:
quality["missing_values"] = int(data.isna().sum().sum())  # count of empty cells
quality["duplicate_rows"] = int(data.duplicated().sum())  # count of fully repeated rows
quality["duplicate_timestamps"] = int(data["timestamp"].duplicated().sum())  # same time appearing twice
quality["is_time_sorted"] = bool(data["timestamp"].is_monotonic_increasing)  # True if time never goes backward

print(quality)  # show the results so far

### 2c. How regular is the sampling?

The time between two readings is the first thing to look at. A regular sensor has one dominant interval; here we count the most common ones.

In [ ]:
interval = data["timestamp"].diff().dropna()  # seconds since the previous reading

quality["interval_median_s"] = float(interval.median())  # median interval
quality["interval_max_s"] = float(interval.max())  # longest interval
quality["intervals_over_1h"] = int((interval > 3600).sum())  # how many gaps longer than 1 hour

top_intervals = interval.value_counts(normalize=True).head(5)  # 5 most common intervals as a share
quality["top_intervals_share"] = {  # save them in the profile
    int(k): round(float(v), 4)  # interval in seconds -> share of all intervals
    for k, v in top_intervals.items()  # go through the 5 items
}  # end of dictionary

print(top_intervals)  # show the 5 most common intervals

### 2d. Gaps and coverage

In [ ]:
# Longest gap without data, computed from the data
gap_end_row = int(interval.idxmax())  # row where the longest gap ends
gap_start_time = pd.to_datetime(data.loc[gap_end_row - 1, "timestamp"], unit="s")  # last reading before the gap
gap_end_time = pd.to_datetime(data.loc[gap_end_row, "timestamp"], unit="s")  # first reading after the gap
quality["largest_gap_days"] = round(float(interval.max() / 86400), 1)  # longest gap in days
quality["largest_gap_start"] = str(gap_start_time)  # last reading before the longest gap
quality["largest_gap_end"] = str(gap_end_time)  # first reading after the longest gap

biggest_gaps = interval.sort_values(ascending=False).head(5)  # the 5 longest intervals in seconds
print(biggest_gaps / 86400)  # convert seconds to days (86400 s = 1 day); the index is the row where each gap ends

# Coverage and the two interval types
all_days = pd.to_datetime(data["timestamp"], unit="s").dt.date  # calendar date of every reading
quality["days_with_data"] = int(all_days.nunique())  # days that have at least one reading
quality["coverage_ratio"] = round(quality["days_with_data"] / quality["span_days"], 3)  # share of days that have data
quality["share_active_intervals"] = round(float((interval <= 2).mean()), 4)  # intervals of 1-2 s (flowing)
quality["share_idle_intervals"] = round(float(((interval >= 300) & (interval <= 302)).mean()), 4)  # intervals of 300-302 s (idle)
quality["share_other_intervals"] = round(1 - quality["share_active_intervals"] - quality["share_idle_intervals"], 4)  # everything else
quality["regularity"] = "event-driven (inferred from interval shares, not documented)"  # type of sampling pattern
profile["context"]["nominal_sampling"] = "1 s documented; observed: 1-2 s when flowing, ~300 s when idle"  # replace the old entry

print(quality)  # show the quality section

**Plot: time between two readings.** Almost all intervals fall into just two groups.

In [ ]:
labels = ["1-2 s\n(flowing)", "300-302 s\n(idle)", "other"]  # three groups of intervals
heights = [  # share of each group in percent
    quality["share_active_intervals"] * 100,  # intervals of 1-2 s
    quality["share_idle_intervals"] * 100,  # intervals of 300-302 s
    quality["share_other_intervals"] * 100,  # everything else
]  # end of list

plt.figure(figsize=(6, 4))  # size of the chart
bars = plt.bar(labels, heights)  # draw the bars
plt.bar_label(bars, fmt="%.1f%%")  # print the percentage above each bar
plt.ylabel("Share of all intervals (%)")  # label of the vertical axis
plt.ylim(0, 65)  # leave room for the labels
plt.title("Time between two readings")  # chart title
save_and_show("plot3_intervals.png")  # fit, save and show the chart

**Plot: which days have data?** Green days have readings, white days do not. (Section 10 checks whether the other fixtures have the same gaps.)

In [ ]:
local_time = pd.to_datetime(data["timestamp"], unit="s").dt.tz_localize("UTC").dt.tz_convert(TZ)  # local time of every reading (assumed time zone)
days_naive = local_time.dt.tz_localize(None).dt.normalize().drop_duplicates()  # one entry per calendar day that has data
total_days = len(days_naive)  # number of days with data
days_per_weekday = days_naive.dt.dayofweek.value_counts().reindex(range(7), fill_value=0)  # days with data per weekday (0 = Monday)
print(total_days, "days with data")  # how many days we divide by later

In [ ]:
all_days = pd.date_range(days_naive.min(), days_naive.max(), freq="D")  # every calendar day in the period
has_data = all_days.isin(days_naive.values)  # True when the day has readings
cal = pd.DataFrame({"month": all_days.to_period("M").astype(str), "day": all_days.day, "has": has_data.astype(int)})  # month, day number, data flag
grid = cal.pivot(index="month", columns="day", values="has")  # one row per month, one column per day number

plt.figure(figsize=(10, 5))  # size of the chart
plt.imshow(grid.values, aspect="auto", cmap="Greens", vmin=0, vmax=1)  # green = data, white = no data
plt.yticks(range(len(grid)), grid.index)  # month names
plt.xticks(range(0, 31, 5), range(1, 32, 5))  # day numbers
plt.xlabel("Day of month")  # label of the horizontal axis
plt.title("A3: days with data (green) and without (white)")  # chart title
save_and_show("A3_calendar.png")  # fit, save and show the chart

**Findings (measured).** No missing values, duplicates or unsorted times. But the stream is **not** sampled once per second: 45.4% of intervals are 1–2 s and 53.3% are 300–302 s (only 1.3% are anything else). Only 358 of 629 days have data; the longest gap is 247.4 days (2019-10-28 to 2020-07-02) and 64 gaps exceed one hour.

**Assumption.** The 300 s readings are a "heartbeat" while idle. The cause of the long gaps is unknown.

## 3. What do the values look like?

Three simple questions: how much of the time is the sensor reading zero, how strong is the flow when it is not, and are there values that look suspicious?

The share of **zero readings** can be counted in two ways: by rows (every reading counts once) or by time (every reading counts for the time until the next one). Because readings are dense while water flows, the two numbers differ a lot.

In [ ]:
profile["value_distribution"] = {}  # new section for value distribution
dist = profile["value_distribution"]  # short name for that section

dist["zero_fraction_by_rows"] = round(float((values == 0).mean()), 4)  # share of readings equal to zero

duration = data["timestamp"].diff().shift(-1)  # seconds until the next reading
valid = duration <= 3600  # ignore gaps longer than 1 hour (and the last row)
zero_time = duration[valid & (values == 0)].sum()  # seconds spent at zero
total_time = duration[valid].sum()  # total seconds counted
dist["zero_fraction_by_time"] = round(float(zero_time / total_time), 4)  # share of time at zero (assumes the value holds until the next reading)

top_values = values.value_counts(normalize=True).head(10)  # 10 most common values as a share
dist["top_values_share"] = {int(k): round(float(v), 4) for k, v in top_values.items()}  # save them in the profile

nonzero = values[values > 0]  # only readings with flow
dist["nonzero_median"] = float(nonzero.median())  # typical flow when water runs
dist["nonzero_p95"] = float(nonzero.quantile(0.95))  # 95% of flowing readings are below this

print(dist)  # show the section

In [ ]:
vd = profile["value_distribution"]  # results about the value distribution
labels = ["share of rows", "share of time"]  # two ways to count
heights = [vd["zero_fraction_by_rows"] * 100, vd["zero_fraction_by_time"] * 100]  # zero share in percent

plt.figure(figsize=(5, 4))  # size of the chart
bars = plt.bar(labels, heights)  # draw the two bars
plt.bar_label(bars, fmt="%.1f%%")  # print the percentage above each bar
plt.ylabel("Readings equal to zero (%)")  # label of the vertical axis
plt.ylim(0, 110)  # leave room for the labels
plt.title("Zero by rows vs zero by time")  # chart title
save_and_show("plot2_zero_share.png")  # fit, save and show the chart

**Plot: how strong is the flow when water runs?** Histogram of all non-zero readings.

In [ ]:
plt.figure(figsize=(8, 3.8))  # size of the chart
plt.hist(nonzero, bins=60)  # histogram of non-zero readings
plt.axvline(dist["nonzero_median"], color="black", linestyle="--", label=f"median = {dist['nonzero_median']:.0f} ml/s")  # typical flow
plt.xlabel("Flow (ml/s)")  # label of the horizontal axis
plt.ylabel("Number of readings")  # label of the vertical axis
plt.title("Non-zero readings")  # chart title
plt.legend()  # show the median line label
save_and_show("plot_nonzero_values.png")  # fit, save and show the chart

**Tails and measurement limits.** Rare large values, and readings outside the vendor working range of the sensor type (1–30 L/min, about 17–500 ml/s). The log scale on the vertical axis makes the tail visible. A reading below the lower limit is not necessarily wrong, but it is outside the range where the sensor is specified.

In [ ]:
to_l_per_min = 60 / 1000  # conversion: ml/s to litres per minute
quantiles = nonzero.quantile([0.5, 0.9, 0.95, 0.99, 0.999])  # typical and large flows among the non-zero readings
print((quantiles * to_l_per_min).round(2).rename("L/min"))  # the same quantiles in L/min
print("largest value:", int(values.max()), "ml/s =", round(values.max() * to_l_per_min, 1), "L/min")  # largest reading

dist["share_nonzero_below_sensor_range"] = round(float((nonzero < SENSOR_MIN_ML_S).mean()), 4)  # share of flowing readings below 1 L/min
dist["share_nonzero_above_sensor_range"] = round(float((nonzero > SENSOR_MAX_ML_S).mean()), 4)  # share of flowing readings above 30 L/min
dist["readings_within_5pct_of_max"] = int((values >= 0.95 * values.max()).sum())  # saturation check: many readings stuck near the maximum would be suspicious
print({k: dist[k] for k in ("share_nonzero_below_sensor_range", "share_nonzero_above_sensor_range", "readings_within_5pct_of_max")})  # show the checks
print(values.value_counts().sort_index(ascending=False).head(5))  # the five largest values and how often they occur

plt.figure(figsize=(8, 3.8))  # size of the chart
plt.hist(nonzero, bins=80)  # histogram of non-zero readings
plt.yscale("log")  # log scale so the rare large values are visible
plt.axvline(SENSOR_MIN_ML_S, color="red", linestyle="--", label="vendor lower limit (1 L/min)")  # lower end of the specified range
plt.axvline(SENSOR_MAX_ML_S, color="red", linestyle=":", label="vendor upper limit (30 L/min)")  # upper end of the specified range
plt.xlim(0, SENSOR_MAX_ML_S * 1.05)  # show the whole specified range
plt.xlabel("Flow (ml/s)")  # label of the horizontal axis
plt.ylabel("Number of readings (log scale)")  # label of the vertical axis
plt.title("Tail of the flow distribution and the sensor working range")  # chart title
plt.legend()  # show which line is which
save_and_show("plot_tail_and_range.png")  # fit, save and show the chart

Finally the basic format of the values: whole numbers or decimals, range, number of different values, bits needed to hold the largest one.

In [ ]:
values = data["value"]  # short name for the value column
profile["value_format"] = {}  # new section for value format results
fmt = profile["value_format"]  # short name for that section

fmt["dtype_in_file"] = str(values.dtype)  # data type pandas detected
fmt["all_integers"] = bool((values % 1 == 0).all())  # True if no decimals anywhere
fmt["min"] = float(values.min())  # smallest value
fmt["max"] = float(values.max())  # largest value
fmt["unique_values"] = int(values.nunique())  # how many different values appear
fmt["negative_values"] = int((values < 0).sum())  # count of values below zero

fmt["bits_needed_unsigned"] = int(values.max()).bit_length()  # bits to store the max value (no negatives)

print(fmt)  # show the results

**Findings (measured).** Zero in 53.9% of rows but 98.75% of the *time*. When water flows the median is 85 ml/s (95th percentile 119). The values are whole numbers from 0 to 382, only 149 different ones. The largest value, 382 ml/s, is about 23 L/min, inside the vendor range (it is far from the 30 L/min upper limit). The printed share of readings below the lower limit and the saturation check show how much of the data lies outside the specified range.

**Assumption.** The zero share by time assumes each value holds until the next reading and ignores gaps over one hour. The sensor working range is a vendor specification for the sensor type, not for this unit.

## 4. Events: periods of flow

A series that is mostly zero is easier to understand as a list of **events**. Our definition: an event is a run of readings with value > 0, split at gaps longer than 60 s. We sort events by duration: *micro* (< 10 s), *medium* (10–60 s) and *sustained* (≥ 60 s). Volume is approximate (flow × time to the next reading, capped at 5 s). Single-row events have duration 0 by construction.

In [ ]:
ev = data.copy()  # work on a copy so the original table stays unchanged
ev["time"] = pd.to_datetime(ev["timestamp"], unit="s")  # readable time (UTC)
ev["dt_prev"] = ev["timestamp"].diff()  # seconds since the previous reading
ev["dt_next"] = ev["timestamp"].diff().shift(-1)  # seconds until the next reading
ev["active"] = ev["value"] > 0  # True when water is flowing

prev_active = ev["active"].shift(1, fill_value=False)  # was the previous reading active?
new_event = ev["active"] & (~prev_active | (ev["dt_prev"] > 60))  # event starts when flow begins or after a gap over 60 s
ev["event_id"] = new_event.cumsum()  # running number of events
active_rows = ev[ev["active"]].copy()  # keep only readings with flow
print(len(active_rows), active_rows["event_id"].nunique())  # active rows and number of events

In [ ]:
active_rows["volume_ml"] = active_rows["value"] * active_rows["dt_next"].clip(upper=5).fillna(1)  # flow x seconds, capped at 5 s so idle gaps are not counted

events = active_rows.groupby("event_id").agg(  # one row per event
    start=("time", "first"),  # event start time
    end=("time", "last"),  # event end time
    rows=("value", "size"),  # number of readings in the event
    mean_flow=("value", "mean"),  # average flow in ml/s
    max_flow=("value", "max"),  # peak flow in ml/s
    volume_ml=("volume_ml", "sum"),  # estimated water volume in ml
)  # end of aggregation
events["duration_s"] = (events["end"] - events["start"]).dt.total_seconds()  # seconds from first to last reading
events["volume_l"] = events["volume_ml"] / 1000  # volume in liters
events.describe()  # summary of all event columns

In [ ]:
profile["events"] = {}  # new section for event results
evp = profile["events"]  # short name for that section

evp["definition"] = "event = consecutive readings with value > 0, split by gaps over 60 s"  # how we defined an event
evp["event_count"] = int(len(events))  # number of events
evp["events_per_active_day"] = round(len(events) / quality["days_with_data"], 2)  # average events per day with data
evp["duration_s_median"] = float(events["duration_s"].median())  # typical event length in seconds
evp["duration_s_p95"] = float(events["duration_s"].quantile(0.95))  # 95% of events are shorter than this
evp["duration_s_max"] = float(events["duration_s"].max())  # longest event in seconds
evp["share_events_under_10s"] = round(float((events["duration_s"] < 10).mean()), 4)  # share of very short events
evp["rows_per_event_median"] = float(events["rows"].median())  # typical number of readings per event
evp["volume_l_median"] = round(float(events["volume_l"].median()), 2)  # typical water volume in liters
evp["volume_l_p95"] = round(float(events["volume_l"].quantile(0.95)), 2)  # 95% of events use less than this
evp["rows_share_in_events_over_60s"] = round(float(events.loc[events["duration_s"] > 60, "rows"].sum() / events["rows"].sum()), 4)  # share of flow readings that sit in long events

print(json.dumps(evp, indent=2))  # show the event section

In [ ]:
events["event_class"] = pd.cut(  # label each event by its duration
    events["duration_s"],  # duration of each event in seconds
    bins=[-1, 10, 60, 10**6],  # edges: under 10 s, 10-60 s, 60 s and above
    labels=["micro", "medium", "sustained"],  # names of the three classes
    right=False,  # left edge included, right edge excluded
)  # end of classification

class_summary = events.groupby("event_class", observed=True).agg(  # one row per class
    events=("rows", "size"),  # number of events in the class
    rows=("rows", "sum"),  # number of readings in the class
    median_flow=("mean_flow", "median"),  # typical average flow in ml/s
    median_volume_l=("volume_l", "median"),  # typical volume in liters
    total_volume_l=("volume_l", "sum"),  # total volume in liters
)  # end of aggregation
class_summary["rows_share"] = class_summary["rows"] / class_summary["rows"].sum()  # share of all flow readings
class_summary.round(3)  # show the table

In [ ]:
micro = events[events["event_class"] == "micro"]  # only the very short events
print(micro["max_flow"].value_counts().head(5))  # the 5 most common peak values in micro events

sustained = events[events["event_class"] == "sustained"]  # only the long events
sustained_hour = sustained["start"].dt.tz_localize("UTC").dt.tz_convert("Europe/Rome").dt.hour  # start hour in local time (assumed)
print(sustained_hour.value_counts(normalize=True).sort_index().round(3))  # share of events per hour

In [ ]:
profile["events"]["class_summary"] = json.loads(class_summary.round(3).to_json())  # convert the table to plain JSON data
profile["events"]["duration_note"] = "single-row events have duration 0 by construction; true duration unknown"  # record the limitation
profile["events"]["volume_note"] = "volume = flow x time to next reading, capped at 5 s (approximate)"  # record how volume was estimated

In [ ]:
sus = events[events["event_class"] == "sustained"].copy()  # only the long events
sus["local_start"] = sus["start"].dt.tz_localize("UTC").dt.tz_convert(TZ)  # event start in local time
sus["hour"] = sus["local_start"].dt.hour  # hour of the day
sus["weekday"] = sus["local_start"].dt.dayofweek  # 0 = Monday, 6 = Sunday
sus["is_weekend"] = sus["weekday"] >= 5  # True on Saturday and Sunday
groups = active_rows.groupby("event_id")["value"]  # flow readings grouped per event
print(len(sus), "sustained events")  # number of long events

**Plot: one event, close up.** One dot is one reading: dense while water flows, sparse around it.

In [ ]:
sustained_ids = events.index[events["event_class"] == "sustained"]  # ids of the long events
one_id = sustained_ids[10]  # pick one long event (change 10 to see another one)
rows_in_event = active_rows[active_rows["event_id"] == one_id]  # flow readings of that event
first_row = rows_in_event.index.min() - 3  # start a few rows before the event
last_row = rows_in_event.index.max() + 3  # end a few rows after the event
window = data.loc[first_row:last_row]  # the readings to draw

x = pd.to_datetime(window["timestamp"], unit="s")  # convert Unix seconds to dates
plt.figure(figsize=(10, 4))  # size of the chart
plt.plot(x, window["value"], marker=".", linewidth=1)  # one dot per reading, joined by a line
plt.xlabel("Time")  # label of the horizontal axis
plt.ylabel("Flow (ml/s)")  # label of the vertical axis
plt.title("One shower event: one dot = one reading")  # chart title
save_and_show("plot1_one_event.png")  # fit, save and show the chart

**Plot: the typical shape of a long event.** Every long event is stretched to the same length (by number of readings, not seconds). Thin lines are single events, the band is the middle 50%, the thick line is the median.

In [ ]:
x_new = np.linspace(0, 1, 100)  # 100 points from event start to event end
curves = []  # list that collects one resampled curve per event
for ev_id in sus.index:  # go through every long event
    v = groups.get_group(ev_id).values  # flow readings of this event
    curves.append(np.interp(x_new, np.linspace(0, 1, len(v)), v))  # stretch or shrink to 100 points
curves = np.array(curves)  # table: events x 100 points

plt.figure(figsize=(10, 4))  # size of the chart
plt.plot(x_new * 100, curves[:60].T, color="gray", alpha=0.2, linewidth=0.8)  # 60 single events, faint
plt.fill_between(x_new * 100, np.percentile(curves, 25, axis=0), np.percentile(curves, 75, axis=0), alpha=0.3, label="middle 50% of events")  # spread
plt.plot(x_new * 100, np.median(curves, axis=0), linewidth=2.5, label="median")  # typical shape
plt.xlabel("Position in event (%), in number of readings")  # label of the horizontal axis
plt.ylabel("Flow (ml/s)")  # label of the vertical axis
plt.title(f"B1: shape of a long event ({len(curves)} events)")  # chart title
plt.legend()  # show which is which
save_and_show("B1_event_shape.png")  # fit, save and show the chart

**Plot: how big are the long events?** Duration, approximate volume and mean flow of the sustained events, plus how many long events happen on a day with data. A strongly skewed histogram means a few very long or very large uses next to many ordinary ones.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))  # three charts side by side
axes[0].hist(sus["duration_s"] / 60, bins=40)  # duration in minutes
axes[0].set_xlabel("Duration (minutes)")  # label of the horizontal axis
axes[0].set_ylabel("Number of events")  # label of the vertical axis
axes[1].hist(sus["volume_l"], bins=40)  # approximate volume
axes[1].set_xlabel("Volume (litres, approximate)")  # label of the horizontal axis
axes[2].hist(sus["mean_flow"], bins=40)  # average flow of each event
axes[2].set_xlabel("Mean flow (ml/s)")  # label of the horizontal axis
fig.suptitle("Long events: how long, how much water, how strong?")  # overall title
save_and_show("plot_event_sizes.png")  # fit, save and show the chart
print(sus[["duration_s", "volume_l", "mean_flow"]].describe().round(1))  # numbers behind the histograms

event_day = sus["local_start"].dt.tz_localize(None).dt.normalize()  # calendar day of each long event
events_per_day = event_day.value_counts().reindex(days_naive.values, fill_value=0)  # long events per day with data (0 if none)
print(events_per_day.value_counts().sort_index())  # how many days have 0, 1, 2, ... long events

plt.figure(figsize=(6, 3.6))  # size of the chart
events_per_day.value_counts().sort_index().plot.bar()  # number of days for each count of long events
plt.xlabel("Long events on one day")  # label of the horizontal axis
plt.ylabel("Number of days with data")  # label of the vertical axis
plt.title("Long events per day")  # chart title
save_and_show("plot_events_per_day.png")  # fit, save and show the chart

**Plot: are readings missing inside an event?** While water flows the sensor should write about one reading per second. Here we count the time between consecutive readings *inside* events, and the completeness of each long event (number of readings divided by duration in seconds plus one). Intervals of 2 s or more mean that a second was skipped or the timestamp was rounded.

In [ ]:
act = active_rows.copy()  # flow readings with their event number
act["gap_in_event"] = act.groupby("event_id")["timestamp"].diff()  # seconds since the previous reading, inside the same event only
inner = act["gap_in_event"].dropna()  # drop the first reading of each event

quality["inside_events_share_1s"] = round(float((inner == 1).mean()), 4)  # share of exactly 1 s intervals inside events
quality["inside_events_share_2s"] = round(float((inner == 2).mean()), 4)  # share of exactly 2 s intervals inside events
quality["inside_events_share_3s_or_more"] = round(float((inner >= 3).mean()), 4)  # share of longer intervals inside events

sus_complete = sus["rows"] / (sus["duration_s"] + 1)  # readings per second of event duration for long events
quality["sustained_completeness_median"] = round(float(sus_complete.median()), 3)  # typical completeness
print({k: quality[k] for k in ("inside_events_share_1s", "inside_events_share_2s", "inside_events_share_3s_or_more", "sustained_completeness_median")})  # show the numbers

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))  # two charts side by side
inner.clip(upper=6).value_counts().sort_index().plot.bar(ax=axes[0])  # intervals inside events, 6 stands for 6 s or more
axes[0].set_xlabel("Seconds between readings inside events (6 = 6 or more)")  # label of the horizontal axis
axes[0].set_ylabel("Number of intervals")  # label of the vertical axis
axes[1].hist(sus_complete, bins=40)  # completeness of each long event
axes[1].set_xlabel("Completeness of long events (1 = every second recorded)")  # label of the horizontal axis
axes[1].set_ylabel("Number of events")  # label of the vertical axis
fig.suptitle("Recording quality inside flow events")  # overall title
save_and_show("plot_recording_quality.png")  # fit, save and show the chart

**Plot: are the tiny "micro" events close to long events?** Distance in seconds from each micro event to the nearest long event. This shows closeness only, not cause.

In [ ]:
micro = events[events["event_class"] == "micro"][["start"]].sort_values("start")  # very short events
sus_end = sus[["end"]].sort_values("end")  # end times of long events
sus_begin = sus[["start"]].rename(columns={"start": "sus_start"}).sort_values("sus_start")  # start times of long events

prev = pd.merge_asof(micro, sus_end, left_on="start", right_on="end", direction="backward")  # nearest long event that ended before
nxt = pd.merge_asof(micro, sus_begin, left_on="start", right_on="sus_start", direction="forward")  # nearest long event that starts after
after_prev = (prev["start"] - prev["end"]).dt.total_seconds().values  # seconds since the previous long event ended
before_next = (nxt["sus_start"] - nxt["start"]).dt.total_seconds().values  # seconds until the next long event starts
nearest = np.fmin(after_prev, before_next)  # distance to the closest long event

plt.figure(figsize=(8, 4))  # size of the chart
plt.hist(np.clip(nearest[~np.isnan(nearest)], 1, None), bins=np.logspace(0, 7, 40))  # histogram with logarithmic bins
plt.xscale("log")  # log scale because distances vary a lot
plt.xlabel("Seconds to the nearest long event")  # label of the horizontal axis
plt.ylabel("Number of micro events")  # label of the vertical axis
plt.title("B4: are micro events close to long events?")  # chart title
save_and_show("B4_micro_distance.png")  # fit, save and show the chart

for limit in [60, 300, 3600]:  # distance limits in seconds
    print(f"within {limit:>5} s of a long event:", round(float(np.nanmean(nearest <= limit)), 3))  # share of micro events that are that close

**Findings (measured).** 903 events: 571 micro, 30 medium, 302 sustained. The sustained events hold 98.4% of all flowing readings; their typical volume is about 19 L and typical mean flow 82 ml/s. 57.6% of the micro events have a peak of exactly 2 ml/s, the smallest non-zero value (about 0.12 L/min, below the vendor lower limit of 1 L/min). Over all intervals, 3.3% are exactly 2 s, which is about 7% of the intervals while water flows; the plots above show whether these skipped seconds are spread evenly or concentrated in a few events.

**Interpretation (not proven).** Sustained events are showers; micro events may be drips, residual water or readings below the sensor's specified range. The cause is unknown.

## 5. When is water used?

Using only the long events (micro events would blur the pattern), we look at the hour of the day, the weekday and the month. Rates are *per day with data*, not per calendar day, and local time uses the assumed time zone. With about 300 events in total, single cells of the weekday/hour map are noisy: look for broad bands, not single cells.

In [ ]:
hour_counts = sus["hour"].value_counts().reindex(range(24), fill_value=0)  # number of events per hour, 0 if none
events_per_day = hour_counts / total_days  # average number of events per day with data

plt.figure(figsize=(10, 4))  # size of the chart
plt.bar(range(24), events_per_day.values)  # one bar per hour
plt.xticks(range(0, 24, 2))  # label every second hour
plt.xlabel("Hour of day (local time, assumed)")  # label of the horizontal axis
plt.ylabel("Showers per day with data")  # label of the vertical axis
plt.title("Typical day: when does a long water use start?")  # chart title
save_and_show("plot7_hour_of_day.png")  # fit, save and show the chart

In [ ]:
evp = profile["events"]  # events section of the profile
hour_share = sus["hour"].value_counts(normalize=True).sort_index()  # share of long events per start hour
evp["start_hour_share_sustained"] = {int(k): round(float(v), 3) for k, v in hour_share.items()}  # save in the profile (long events only)
print(hour_share.round(3))  # show the share per hour

**Finding (measured):** two main peaks, about 07–09 h (about 25% of the long events) and 16–19 h (about 36%); only about 3% start between 00 and 06 h.

**Plot: how much water, by start hour** (approximate volume).

In [ ]:
volume_by_hour = sus.groupby("hour")["volume_l"].sum().reindex(range(24), fill_value=0)  # total liters started in each hour
liters_per_day = volume_by_hour / total_days  # average liters per day with data

plt.figure(figsize=(10, 4))  # size of the chart
plt.bar(range(24), liters_per_day.values)  # one bar per hour
plt.xticks(range(0, 24, 2))  # label every second hour
plt.xlabel("Hour of day (local time, assumed)")  # label of the horizontal axis
plt.ylabel("Liters per day with data (approx.)")  # label of the vertical axis
plt.title("Typical day: how much water, by start hour")  # chart title
save_and_show("plot8_volume_by_hour.png")  # fit, save and show the chart

**Plot: weekday vs hour.** Average events per day with data.

In [ ]:
counts = pd.crosstab(sus["weekday"], sus["hour"])  # events for each weekday and hour
counts = counts.reindex(index=range(7), columns=range(24), fill_value=0)  # make sure all 7 days and 24 hours exist
per_day = counts.div(days_per_weekday.reindex(range(7)), axis=0)  # average events per day with data, for each weekday

plt.figure(figsize=(10, 4))  # size of the chart
plt.imshow(per_day.values, aspect="auto", cmap="Blues")  # draw the colour map
plt.colorbar(label="Showers per day with data")  # colour scale
plt.yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])  # weekday names
plt.xticks(range(0, 24, 2))  # label every second hour
plt.xlabel("Hour of day (local time, assumed)")  # label of the horizontal axis
plt.title("Weekday vs hour")  # chart title
save_and_show("plot9_weekday_hour.png")  # fit, save and show the chart

**Plot: weekday vs weekend.** Does the morning peak move at the weekend? The legend shows the days and events behind each line.

In [ ]:
hours = np.arange(24)  # hours of the day
days_wd = days_per_weekday.loc[0:4].sum()  # days with data, Monday to Friday
days_we = days_per_weekday.loc[5:6].sum()  # days with data, Saturday and Sunday
wd = sus[~sus["is_weekend"]]["hour"].value_counts().reindex(hours, fill_value=0)  # weekday events per hour
we = sus[sus["is_weekend"]]["hour"].value_counts().reindex(hours, fill_value=0)  # weekend events per hour

plt.figure(figsize=(10, 4))  # size of the chart
plt.plot(hours, wd.values / days_wd, marker="o", label=f"Mon-Fri ({int(days_wd)} days, {int(wd.sum())} events)")  # weekday line
plt.plot(hours, we.values / days_we, marker="o", label=f"Sat-Sun ({int(days_we)} days, {int(we.sum())} events)")  # weekend line
plt.xticks(range(0, 24, 2))  # label every second hour
plt.xlabel("Hour of day (local time, assumed)")  # label of the horizontal axis
plt.ylabel("Long events per day with data")  # label of the vertical axis
plt.title("A1: weekday vs weekend")  # chart title
plt.legend()  # show which line is which
save_and_show("A1_weekday_weekend.png")  # fit, save and show the chart

**Plot: events per day, by month.** Compare the printed table (days with data per month) before reading the bars: months with few days are less reliable.

In [ ]:
sus["month"] = sus["local_start"].dt.tz_localize(None).dt.to_period("M")  # month of each event
days_per_month = days_naive.dt.to_period("M").value_counts().sort_index()  # days with data per month
events_per_month = sus["month"].value_counts().reindex(days_per_month.index, fill_value=0)  # long events per month
per_day = events_per_month / days_per_month  # events per day with data

plt.figure(figsize=(10, 4))  # size of the chart
plt.bar(range(len(per_day)), per_day.values)  # one bar per month
plt.xticks(range(len(per_day)), per_day.index.astype(str), rotation=60)  # month names under the bars
plt.ylabel("Long events per day with data")  # label of the vertical axis
plt.title("A2: events per day, by month")  # chart title
save_and_show("A2_by_month.png")  # fit, save and show the chart

print(pd.DataFrame({"days_with_data": days_per_month, "long_events": events_per_month}))  # counts behind each bar

**Plot: does the usage change over time?** Typical mean flow and typical volume of the long events, by month. A steady drift would point to changes in the household, the plumbing or the sensor; months with only a few events are not reliable.

In [ ]:
monthly_level = sus.groupby("month").agg(  # one row per month
    events=("rows", "size"),  # number of long events
    median_flow=("mean_flow", "median"),  # typical mean flow in ml/s
    median_volume_l=("volume_l", "median"),  # typical approximate volume in litres
)  # end of aggregation
print(monthly_level.round(1))  # numbers behind the chart, including events per month

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)  # two charts sharing the month axis
axes[0].plot(range(len(monthly_level)), monthly_level["median_flow"], marker="o")  # typical flow
axes[0].set_ylabel("Median of event mean flow (ml/s)")  # label of the vertical axis
axes[1].plot(range(len(monthly_level)), monthly_level["median_volume_l"], marker="o")  # typical volume
axes[1].set_ylabel("Median event volume (litres, approximate)")  # label of the vertical axis
for ax in axes:  # same month labels on both charts
    ax.set_xticks(range(len(monthly_level)))  # one tick per month
    ax.set_xticklabels(monthly_level.index.astype(str), rotation=60)  # month names
fig.suptitle("Long events by month")  # overall title
save_and_show("plot_monthly_level.png")  # fit, save and show the chart

**Plot: time between long events.** The long tail includes data gaps, so it is not user behaviour.

In [ ]:
sus_sorted = sus.sort_values("start")  # long events in time order
gap_h = ((sus_sorted["start"].shift(-1) - sus_sorted["end"]).dt.total_seconds() / 3600).dropna()  # hours between the end of one event and the start of the next

plt.figure(figsize=(8, 4))  # size of the chart
plt.hist(gap_h.clip(lower=0.01), bins=np.logspace(-2, 4, 40))  # histogram with logarithmic bins
plt.xscale("log")  # log scale because gaps vary a lot
plt.xlabel("Hours between long events (includes data gaps)")  # label of the horizontal axis
plt.ylabel("Number of gaps")  # label of the vertical axis
plt.title("A4: time between long events")  # chart title
save_and_show("A4_gaps.png")  # fit, save and show the chart
print(gap_h.describe())  # summary of the gaps in hours

**Interpretation (not proven).** The two daily peaks match a typical household rhythm (morning and evening), which supports the idea that sustained events are showers. The time zone is an assumption.

---

# Part B. Looking deeper

## 6. How predictable is the signal?

From here on we ask: **if we know what came before, how well can we guess the next reading?** This tells us how much structure the sensor stream has.

Tests: run lengths (how long does a value stay the same?), correlation with the previous reading, two simple guesses, and **entropy**.

*Entropy in plain words:* the average number of yes/no questions needed to guess a value. A value that is almost always 0 needs very few questions (low entropy, easy to guess); a value that could be anything needs many (high entropy, hard to guess). Entropy depends on what we already know: knowing the previous value can make the next one much easier to guess.

**Plot: previous value vs current value.** Each cell counts pairs of consecutive readings (pairs 0 → 0 are left out because they dominate). Points near the diagonal mean "the next value is close to the previous one".

In [ ]:
pairs = pd.DataFrame({"prev": values.shift(1), "cur": values}).dropna()  # consecutive pairs of readings
pairs = pairs[(pairs["prev"] > 0) | (pairs["cur"] > 0)]  # leave out 0 -> 0 pairs
top = float(values.max())  # largest value, for the diagonal line

plt.figure(figsize=(5.5, 5))  # size of the chart
hb = plt.hexbin(pairs["prev"], pairs["cur"], gridsize=60, bins="log", mincnt=1, cmap="Blues")  # density of pairs
plt.plot([0, top], [0, top], color="gray", linewidth=1)  # diagonal: current = previous
plt.colorbar(hb, label="Number of pairs (log scale)")  # colour scale
plt.xlabel("Previous value (ml/s)")  # label of the horizontal axis
plt.ylabel("Current value (ml/s)")  # label of the vertical axis
plt.title("Consecutive readings")  # chart title
save_and_show("plot_lag.png")  # fit, save and show the chart

**Runs.** How long does a value repeat?

In [ ]:
profile["predictability"] = {}  # new section for predictability results
pred = profile["predictability"]  # short name for that section

run_id = (values != values.shift()).cumsum()  # new id every time the value changes
run_lengths = values.groupby(run_id).size()  # number of rows in each run

pred["run_count"] = int(len(run_lengths))  # how many runs exist
pred["run_length_mean"] = round(float(run_lengths.mean()), 2)  # average run length
pred["run_length_median"] = float(run_lengths.median())  # typical run length
pred["run_length_max"] = int(run_lengths.max())  # longest run
pred["rows_in_runs_of_2_or_more"] = round(float(run_lengths[run_lengths >= 2].sum() / len(values)), 4)  # share of rows that repeat the previous value

run_value = values.groupby(run_id).first()  # the value of each run
pred["zero_run_length_mean"] = round(float(run_lengths[run_value == 0].mean()), 2)  # average length of runs of zero
pred["nonzero_run_length_mean"] = round(float(run_lengths[run_value > 0].mean()), 2)  # average length of runs with flow

print(pred)  # show results so far

**Correlation with the previous readings.**

In [ ]:
pred["autocorr_lag1"] = round(float(values.autocorr(lag=1)), 4)  # correlation with the previous reading
pred["autocorr_lag2"] = round(float(values.autocorr(lag=2)), 4)  # correlation with the reading two rows back

print(pred["autocorr_lag1"], pred["autocorr_lag2"])  # show both values

**Two simple guesses.** Guess 1: "the next value equals the previous one". Guess 2: "the signal continues in the same direction" (a straight line).

In [ ]:
prev1 = values.shift(1)  # previous value
prev2 = values.shift(2)  # value two rows back

guess_same = prev1  # guess 1: next value equals the previous one
guess_line = 2 * prev1 - prev2  # guess 2: continue the same direction (straight line)

error_same = (values - guess_same).dropna()  # error of guess 1
error_line = (values - guess_line).dropna()  # error of guess 2

pred["guess_same_exact_share"] = round(float((error_same == 0).mean()), 4)  # share of exactly right guesses, guess 1
pred["guess_line_exact_share"] = round(float((error_line == 0).mean()), 4)  # share of exactly right guesses, guess 2
pred["entropy_error_same"] = round(entropy_bits(error_same), 3)  # entropy of the errors, guess 1
pred["entropy_error_line"] = round(entropy_bits(error_line), 3)  # entropy of the errors, guess 2

print(pred)  # show results so far

**Entropy of the values and of their changes.**

In [ ]:
dist["entropy_bits_value"] = round(entropy_bits(values), 3)  # entropy of the raw values
dist["entropy_bits_diff1"] = round(entropy_bits(values.diff().dropna()), 3)  # entropy of the changes between readings
dist["max_possible_entropy_bits"] = round(float(np.log2(values.nunique())), 3)  # reference line: all distinct values equally likely (worst case)

print(dist["entropy_bits_value"], dist["entropy_bits_diff1"], dist["max_possible_entropy_bits"])  # show the numbers

**Entropy when the previous value is known.**

In [ ]:
pair = pd.DataFrame({"prev": prev1, "curr": values}).dropna()  # table of (previous, current) pairs
pair_key = pair["prev"].astype(int) * 1000 + pair["curr"].astype(int)  # one number that encodes each pair

h_pair = entropy_bits(pair_key)  # entropy of the pair together
h_prev = entropy_bits(pair["prev"])  # entropy of the previous value alone
pred["entropy_value_given_prev"] = round(h_pair - h_prev, 3)  # bits still needed once the previous value is known
pred["distinct_pairs"] = int(pair_key.nunique())  # how many different pairs appear (many pairs = less reliable estimate)

print(pred["entropy_value_given_prev"], pred["distinct_pairs"])  # show both values

**Plot: the entropy ladder.** The same data, read in different ways. Shorter bars are easier to guess.

In [ ]:
labels = [  # four ways of reading the same data
    f"Worst case\n({values.nunique()} values equally likely)",
    "Value alone",
    "Change from the previous value",
    "Value, given the previous value",
]  # end of labels
bits = [dist["max_possible_entropy_bits"], dist["entropy_bits_value"], dist["entropy_bits_diff1"], pred["entropy_value_given_prev"]]  # entropy in bits

plt.figure(figsize=(8, 3.8))  # size of the chart
bars = plt.barh(labels[::-1], bits[::-1])  # horizontal bars, worst case on top
plt.bar_label(bars, fmt="%.2f bit")  # print the value at the end of each bar
plt.xlabel("Entropy (bits per reading)")  # label of the horizontal axis
plt.title("How hard is it to guess a reading?")  # chart title
plt.xlim(0, max(bits) * 1.2)  # leave room for the labels
save_and_show("plot_entropy_ladder.png")  # fit, save and show the chart

**Findings (measured).**
- Zero runs average 109 rows, non-zero runs 2.3 rows. Lag-1 autocorrelation is 0.992.
- "Same as the previous value" is exactly right 79.5% of the time; "continue the same direction" is worse (70.2%) because the signal is step-like.
- Entropy drops from 3.50 bit (value alone) to 1.34 bit (change from the previous value) and 0.88 bit (value given the previous value). The worst case for 149 values is 7.22 bit; it is only a reference line.

**Limitation.** The 0.88 bit estimate uses 1,866 distinct (previous, current) pairs from the same data, so it is probably a little optimistic.

## 7. The values sit on a fixed step grid

The distinct values look like multiples of about 2.25 (2, 4, 6, 9, 11, 13, ...). If so, the sensor does not produce free numbers but counts steps: `value = floor(2.25 × k)` for a whole number k. We test the rule on every reading.

In [ ]:
unique_values = np.sort(values.unique())  # all distinct values, sorted
steps = np.diff(unique_values)  # gaps between neighbouring distinct values

print(unique_values[:25])  # the 25 smallest distinct values
print(pd.Series(steps).value_counts().head(5))  # the most common gaps between distinct values

In [ ]:
pulse = np.ceil(values * 4 / 9)  # pulse count k that gives each value under the rule value = floor(2.25 * k)
rebuilt = np.floor(pulse * 2.25)  # apply the rule forward to rebuild the values

match = (rebuilt == values)  # True where the rule reproduces the value exactly
fmt["grid_rule_match_share"] = round(float(match.mean()), 6)  # share of readings that fit the rule
fmt["grid_rule_unmatched_unique"] = sorted(values[~match].unique().tolist())[:20]  # up to 20 distinct values that do not fit
fmt["pulse_count_max"] = int(pulse.max())  # largest pulse count
fmt["pulse_count_bits"] = int(pulse.max()).bit_length()  # bits needed to store the pulse count
fmt["entropy_bits_pulse"] = round(entropy_bits(pulse), 3)  # entropy of pulse counts (should equal the value entropy)
fmt["entropy_bits_pulse_diff"] = round(entropy_bits(pulse.diff().dropna()), 3)  # entropy of changes in pulse count
fmt["entropy_bits_value_diff"] = dist["entropy_bits_diff1"]  # earlier entropy of changes in value, for comparison

print(fmt)  # show the updated value format section

In [ ]:
fmt["grid_rule"] = "value = floor(2.25 * k), k = integer pulse count; fits all rows, constant not verified against sensor docs"  # the rule we tested
fmt["possible_pulse_counts"] = int(pulse.max()) + 1  # pulse counts from 0 to max
fmt["unused_pulse_counts"] = int(pulse.max()) + 1 - int(pulse.nunique())  # pulse counts that never appear
fmt["entropy_bits_pulse_diff_gain"] = round(fmt["entropy_bits_value_diff"] - fmt["entropy_bits_pulse_diff"], 3)  # bits saved per value by using pulse changes
print(fmt)  # show the updated section

**Does the sensor specification agree?** The vendor specification of the YF-S201 gives about 2.25 ml per pulse (450 pulses per litre would be 1000/450 = 2.222 ml). The test below checks which constant and which rounding rule can produce the observed values: for each candidate rule we list the observed values that the rule can *not* produce.

In [ ]:
observed = set(int(v) for v in np.unique(values))  # all distinct values in the file
k_all = np.arange(0, 400)  # candidate pulse counts

candidates = [  # constant and rounding rule to test
    ("2.25 ml, round down", 2.25, np.floor),
    ("2.25 ml, round to nearest", 2.25, np.round),
    ("1000/450 ml, round down", 1000 / 450, np.floor),
    ("1000/450 ml, round to nearest", 1000 / 450, np.round),
]  # end of list

rule_check = {}  # results for the profile
for name, constant, rule in candidates:  # test every candidate
    producible = set(int(v) for v in rule(constant * k_all))  # values this rule can produce
    not_explained = sorted(observed - producible)  # observed values the rule cannot produce
    rule_check[name] = len(not_explained)  # keep the count
    print(f"{name:32s} observed values not explained: {len(not_explained):>3}  {not_explained[:6]}")  # show the result

fmt["constant_check_values_not_explained"] = rule_check  # store in the profile

**Findings (measured).** The rule `floor(2.25 × k)` fits 100% of the rows. k reaches 170 and needs 8 bits instead of 9. Entropy of k is the same as for the values (it is only a relabelling), but the entropy of the changes falls from 1.34 to 1.18 bit. The constant check shows which constant and rounding rule can reproduce the observed values.

**Reading the sensor specification (not verified for this unit).** The 2.25 ml per pulse of the vendor specification agrees with the grid, and the largest value (about 23 L/min) is inside the 1–30 L/min working range. That the logger multiplies a pulse count by 2.25 and rounds down is still a hypothesis; the dataset documentation does not say so.

## 8. Noise and levels inside events

While water flows, how much do values move from one reading to the next, and how much of the spread is simply different flow levels in different events?

In [ ]:
sustained_ids = events.index[events["event_class"] == "sustained"]  # ids of the long events
sus_rows = active_rows[active_rows["event_id"].isin(sustained_ids)]  # flow readings that belong to long events
sus_pulse = pulse.loc[sus_rows.index]  # pulse counts of those readings
sus_diff = sus_pulse.groupby(sus_rows["event_id"]).diff().dropna()  # changes between neighbours, inside each event only

profile["noise"] = {}  # new section for noise results
nz = profile["noise"]  # short name for that section

nz["sustained_rows"] = int(len(sus_pulse))  # readings inside long events
nz["pulse_median"] = float(sus_pulse.median())  # typical pulse count while flowing
nz["pulse_std"] = round(float(sus_pulse.std()), 3)  # spread of pulse counts while flowing
nz["entropy_bits_pulse_in_events"] = round(entropy_bits(sus_pulse), 3)  # entropy of pulse counts inside events
nz["entropy_bits_diff_in_events"] = round(entropy_bits(sus_diff), 3)  # entropy of changes inside events
nz["diff_zero_share"] = round(float((sus_diff == 0).mean()), 4)  # share of unchanged readings inside events
nz["diff_within_1_share"] = round(float((sus_diff.abs() <= 1).mean()), 4)  # share of changes of at most one pulse
nz["diff_std"] = round(float(sus_diff.std()), 3)  # spread of the changes

outside_rows = pulse.drop(sus_rows.index)  # all readings that are not in long events
nz["entropy_bits_pulse_outside_events"] = round(entropy_bits(outside_rows), 3)  # entropy of pulse counts outside long events

print(json.dumps(nz, indent=2))  # show the noise section

**Plot: the edges of long events.** The first and last 10 readings of events with at least 20 readings. Edges are where the values change most.

In [ ]:
first10, last10 = [], []  # first and last 10 readings of each event
for ev_id in sus.index:  # go through every long event
    v = groups.get_group(ev_id).values  # flow readings of this event
    if len(v) >= 20:  # only events long enough to have separate start and end
        first10.append(v[:10])  # first 10 readings
        last10.append(v[-10:])  # last 10 readings
first10, last10 = np.array(first10), np.array(last10)  # convert to tables

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)  # two charts side by side
for ax, arr, pos, name in [(axes[0], first10, range(1, 11), "start"), (axes[1], last10, range(-9, 1), "end")]:  # one chart for start, one for end
    ax.fill_between(pos, np.percentile(arr, 25, axis=0), np.percentile(arr, 75, axis=0), alpha=0.3)  # middle 50% of events
    ax.plot(pos, np.median(arr, axis=0), marker="o")  # median at each position
    ax.set_xlabel(f"Reading number at the {name} of the event")  # label of the horizontal axis
axes[0].set_ylabel("Flow (ml/s)")  # label of the vertical axis
fig.suptitle(f"B2: edges of long events ({len(first10)} events)")  # chart title
save_and_show("B2_event_edges.png")  # fit, save and show the chart

**Plot: does each event have its own level?** Each dot is one event's mean flow; the bar is the spread inside the event.

In [ ]:
sus_rows = active_rows[active_rows["event_id"].isin(sus.index)]  # flow readings inside long events
stats = sus_rows.groupby("event_id")["value"].agg(["mean", "std", "size"]).dropna().sort_values("mean")  # mean and spread per event, sorted by mean

plt.figure(figsize=(10, 4))  # size of the chart
plt.errorbar(range(len(stats)), stats["mean"], yerr=stats["std"], fmt="o", markersize=3, alpha=0.6)  # dot = event mean, bar = spread inside the event
plt.xlabel("Events sorted by their mean flow")  # label of the horizontal axis
plt.ylabel("Flow (ml/s)")  # label of the vertical axis
plt.title("B3: each event has its own level?")  # chart title
save_and_show("B3_event_levels.png")  # fit, save and show the chart

print("spread of event means (between events):", round(float(stats["mean"].std()), 2))  # variation between events
print("typical spread inside an event       :", round(float(np.sqrt((stats["std"] ** 2).mean())), 2))  # variation inside events

**Findings (measured).**
- Inside long events 94.3% of the step changes are within ±1 pulse (the sensor's smallest step) and 57.4% are exactly zero, but the tail is heavy (standard deviation of the changes 3.09 pulses).
- Entropy inside events is 5.41 bit for levels and 1.85 bit for changes; outside events it is only 0.15 bit.
- The spread of the event means (between events) is 23.6 ml/s; the typical spread inside an event is 21.2 ml/s. So roughly half of the level variation comes from *different levels in different events*, not from noise.

**Limitation.** The spread of levels includes real differences in flow, so the noise estimates are upper bounds. The vendor accuracy of about ±10% is a calibration limit of the sensor type; it does not tell how noisy single readings are.

## 9. Time as a data stream

Each reading also carries a time. Instead of the raw time we can look at **the interval since the previous reading**. This section looks at that interval stream, and tests the "heartbeat" idea: after a zero reading, does the next reading come after 5 minutes or more?

In [ ]:
ts = data["timestamp"].astype("int64")  # timestamps as whole seconds
delta = ts.diff().fillna(0).astype("int64")  # seconds since the previous reading (first row = 0)

profile["timestamps"] = {}  # new section for timestamp results
tsp = profile["timestamps"]  # short name for that section

tsp["first_timestamp"] = int(ts.iloc[0])  # first time; all later times follow from the intervals
tsp["distinct_deltas"] = int(delta.nunique())  # how many different intervals appear
tsp["delta_top5_share"] = round(float(delta.value_counts(normalize=True).head(5).sum()), 4)  # share of the 5 most common intervals
tsp["entropy_bits_delta"] = round(entropy_bits(delta), 3)  # entropy of the intervals

print(tsp)  # show the timestamp section

In [ ]:
next_delta = delta.shift(-1)  # seconds until the next reading
valid = next_delta.notna()  # the last row has no next reading

check = pd.crosstab(  # count readings for each combination
    (values[valid] == 0).rename("value_is_zero"),  # is the current value zero?
    (next_delta[valid] >= 300).rename("next_gap_300_or_more"),  # does the next reading come after 5 minutes or more?
)  # end of crosstab

print(check)  # counts
print(check.div(check.sum(axis=1), axis=0).round(4))  # share within each row (each value type)

In [ ]:
tsp["next_gap_300_after_zero_share"] = round(float(check.loc[True, True] / check.loc[True].sum()), 4)  # share of zero readings followed by a gap of 300 s or more
tsp["next_gap_short_after_nonzero_share"] = round(float(check.loc[False, False] / check.loc[False].sum()), 4)  # share of nonzero readings followed by a short gap
tsp["heartbeat_note"] = "gap class is almost fully determined by the current value (zero vs nonzero); cause not documented"  # what we observed
print({k: tsp[k] for k in ("next_gap_300_after_zero_share", "next_gap_short_after_nonzero_share")})  # show the two shares

In [ ]:
share = check.div(check.sum(axis=1), axis=0)  # share inside each row of the table
heights = share[True].values * 100  # share of readings followed by a gap of 300 s or more (rows: value>0, value=0)
labels = ["value > 0", "value = 0"]  # the two kinds of current reading

plt.figure(figsize=(5, 4))  # size of the chart
bars = plt.bar(labels, heights)  # draw the bars
plt.bar_label(bars, fmt="%.1f%%")  # print the percentage above each bar
plt.ylabel("Next reading comes after 5 min or more (%)")  # label of the vertical axis
plt.ylim(0, 110)  # leave room for the labels
plt.title("The current value predicts the next gap")  # chart title
save_and_show("plot4_heartbeat.png")  # fit, save and show the chart

**Findings (measured).** Only 421 different intervals occur, and the five most common ones cover 98.7% of all intervals. After a zero reading the next gap is 300 s or more in 99.16% of the cases; after a non-zero reading it is shorter in 98.92%. So the *value* almost fully determines how long the sensor will wait: the value and the time are not independent.

**Interpretation (not proven).** The logger writes a heartbeat about every 5 minutes while idle.

---

# Part C. Other fixtures and summary

## 10. Do the findings hold for the other fixtures?

The same checks are repeated for the other fixtures of the same apartment: coverage (are the gaps shared?), a fingerprint of simple statistics, and how each fixture is used (events). The toilet and dishwasher files are almost empty and skipped. Note that `entropy_value` and `entropy_given_prev` use the raw values, so they are not directly comparable with the pulse-count figures for the shower above.

In [ ]:
names = {  # fixtures to compare (Toilet and Dishwasher are skipped because their files are almost empty)
    "Shower": "feed_Shower.MYD.csv",
    "Washbasin": "feed_Washbasin.MYD.csv",
    "Kitchen faucet": "feed_Kitchenfaucet.MYD.csv",
    "Bidet": "feed_Bidet.MYD.csv",
    "Washing machine": "feed_Washingmachine.MYD.csv",
    "Whole house": "feed_WholeHouse.MYD.csv",
}  # end of dictionary

def load_file(file_name, clean=True):  # open one file as a table, detecting its format
    path = DATA_DIR / file_name  # full path of the file
    with open(path, "r", errors="replace") as f:  # open the file as text
        first_line = f.readline()  # read only the first line to detect the format
    if "," in first_line:  # comma format with a header row (Time,Flow)
        df = pd.read_csv(path, sep=",", header=0, names=["timestamp", "value"])  # comma separator, replace the header with our names
    else:  # space format without a header
        df = pd.read_csv(path, sep=r"\s+", header=None, names=["timestamp", "value"])  # space separator, our own names
    df = df.apply(pd.to_numeric, errors="coerce").dropna()  # force numbers and drop broken rows
    if clean:  # remove impossible timestamps
        n_before = len(df)  # rows before the time check
        df = df[(df["timestamp"] >= MIN_TS) & (df["timestamp"] <= MAX_TS)]  # keep times inside the assumed range
        print(file_name, "| rows dropped for impossible timestamps:", n_before - len(df))  # report what was removed
    return df.reset_index(drop=True)  # clean row numbers

def segment_events(df, split_gap_s=60):  # same event definition as in section 4, for any fixture
    ev = df.copy()  # work on a copy
    ev["time"] = pd.to_datetime(ev["timestamp"], unit="s")  # readable time (UTC)
    ev["dt_prev"] = ev["timestamp"].diff()  # seconds since the previous reading
    ev["dt_next"] = ev["timestamp"].diff().shift(-1)  # seconds until the next reading
    ev["active"] = ev["value"] > 0  # True when water is flowing
    prev_active = ev["active"].shift(1, fill_value=False)  # was the previous reading active?
    new_event = ev["active"] & (~prev_active | (ev["dt_prev"] > split_gap_s))  # event starts when flow begins or after a long gap
    ev["event_id"] = new_event.cumsum()  # running number of events
    act = ev[ev["active"]].copy()  # only readings with flow
    act["volume_ml"] = act["value"] * act["dt_next"].clip(upper=5).fillna(1)  # flow x seconds, capped at 5 s
    ev_table = act.groupby("event_id").agg(  # one row per event
        start=("time", "first"), end=("time", "last"), rows=("value", "size"),  # start, end, number of readings
        mean_flow=("value", "mean"), max_flow=("value", "max"), volume_ml=("volume_ml", "sum"),  # flow and volume
    )  # end of aggregation
    ev_table["duration_s"] = (ev_table["end"] - ev_table["start"]).dt.total_seconds()  # seconds from first to last reading
    ev_table["volume_l"] = ev_table["volume_ml"] / 1000  # volume in litres
    ev_table["event_class"] = pd.cut(ev_table["duration_s"], bins=[-1, 10, 60, 10**9], labels=["micro", "medium", "sustained"], right=False)  # duration classes
    return ev_table  # table of events

def fingerprint_of(df):  # simple statistics of one sensor stream
    v, t = df["value"], df["timestamp"].astype("int64")  # values and times
    nxt = t.diff().shift(-1)  # seconds until the next reading
    ok = nxt <= 3600  # ignore gaps longer than 1 hour
    v_int = np.round(v).astype("int64")  # values as whole numbers
    return {  # one row of the comparison table
        "rows": len(df),  # number of readings
        "days_with_data": int(pd.to_datetime(t, unit="s").dt.normalize().nunique()),  # days with at least one reading (UTC days)
        "all_integers": bool((v % 1 == 0).all()),  # True if no decimals
        "max_value": float(v.max()),  # largest value
        "max_l_per_min": float(v.max() * 60 / 1000),  # largest value in L/min
        "zero_share_rows": float((v == 0).mean()),  # share of zero by rows
        "zero_share_time": float(nxt[ok & (v == 0)].sum() / nxt[ok].sum()),  # share of zero by time
        "share_interval_le_2s": float((t.diff().dropna() <= 2).mean()),  # share of intervals of 2 s or less
        "share_interval_300_302s": float(((t.diff() >= 300) & (t.diff() <= 302)).mean()),  # share of intervals of about 5 minutes
        "entropy_value": entropy_bits(v_int),  # entropy of the values
        "entropy_given_prev": cond_entropy(v_int),  # entropy given the previous value
        "autocorr_lag1": float(v.autocorr(lag=1)),  # correlation with the previous reading
    }  # end of dictionary

In [ ]:
seg_check = segment_events(data)  # events of the shower with the reusable function
print("same event definition as section 4:", len(seg_check) == len(events), "|", len(seg_check), "events")  # the two counts should agree

### Data issues found in the other files

- **Washbasin:** one row has an impossible timestamp (time jumps back by about 50 years). It is dropped by the time filter (`MIN_TS`/`MAX_TS`, an assumption).
- **Kitchen faucet and bidet** use a different file format (comma-separated with a `Time,Flow` header row); `load_file` detects this.
- **Whole house:** the values are huge, alternate in sign and do not look like water flow. The cause is unknown (a different scale or format, or mixed content), so its values are excluded from all comparisons. Its timestamps are used only in the coverage plot.

In [ ]:
raw_wb = load_file("feed_Washbasin.MYD.csv", clean=False)  # washbasin without the time filter
wb_diff = raw_wb["timestamp"].diff()  # seconds since the previous row
bad_row = int(wb_diff.idxmin())  # row where time jumps back the most
print("washbasin: time goes backward", int((wb_diff < 0).sum()), "time(s); sorted:", bool(raw_wb["timestamp"].is_monotonic_increasing))  # how many times and whether sorted
print(raw_wb.iloc[max(bad_row - 2, 0): bad_row + 3])  # rows around the broken timestamp

raw_hh = load_file("feed_WholeHouse.MYD.csv", clean=False)  # whole house without the time filter
print(raw_hh["value"].describe())  # summary of the whole-house values
print("negative values:", int((raw_hh["value"] < 0).sum()), "| values above 32767:", int((raw_hh["value"] > 32767).sum()))  # values that do not look like flow

### 10a. Coverage: are the gaps shared?

If the 247-day gap of the shower also appears in the other fixtures, the logger or the whole system was off (not just unused). If the other fixtures have data in that period, the gap belongs to the shower file only. Each mark is a day with at least one reading (UTC days); the printed counts only use days that lie completely inside the gap.

In [ ]:
frames = {name: load_file(f) for name, f in names.items()}  # load every file with the time filter
for name, df in frames.items():  # check the time order in each file
    print(name, "| sorted in time:", bool(df["timestamp"].is_monotonic_increasing), "| rows:", len(df))  # should be True

fig, ax = plt.subplots(figsize=(11, 3.4))  # size of the chart
for y, (name, df) in enumerate(frames.items()):  # one row of marks per fixture
    day_marks = pd.to_datetime(df["timestamp"], unit="s").dt.normalize().drop_duplicates()  # days with data
    ax.scatter(day_marks, [y] * len(day_marks), marker="|", s=80)  # one mark per day
ax.axvspan(gap_start_time, gap_end_time, color="red", alpha=0.12, label="longest gap of the shower file")  # the shower gap
ax.set_yticks(range(len(frames)))  # one tick per fixture
ax.set_yticklabels(list(frames.keys()))  # fixture names
ax.set_title("Days with data per fixture")  # chart title
ax.legend(loc="lower right")  # explain the red band
save_and_show("plot_coverage_all_fixtures.png")  # fit, save and show the chart

first_full_day = gap_start_time.normalize() + pd.Timedelta(days=1)  # first calendar day completely inside the gap
last_full_day = gap_end_time.normalize() - pd.Timedelta(days=1)  # last calendar day completely inside the gap
n_full_days = (last_full_day - first_full_day).days + 1  # number of days completely inside the gap
for name, df in frames.items():  # days with data inside the shower gap, per fixture
    day_of_reading = pd.to_datetime(df["timestamp"], unit="s").dt.normalize()  # calendar day of every reading (UTC)
    inside = day_of_reading[(day_of_reading >= first_full_day) & (day_of_reading <= last_full_day)].nunique()  # days with data inside the gap
    print(f"{name:16s} days with data inside the shower gap: {inside:>4} of {n_full_days}")  # show the comparison

### 10b. Fingerprint: simple statistics per fixture

Whole house is left out here because its values cannot be read as flow.

In [ ]:
fixtures = [n for n in frames if n != "Whole house"]  # all fixtures except the unreadable whole-house file
fp = pd.DataFrame({n: fingerprint_of(frames[n]) for n in fixtures}).T  # table: one row per fixture
display(fp.round(3))  # show the table

plots = [("zero_share_time", "Zero share (by time)"), ("share_interval_le_2s", "Intervals <= 2 s"),  # metrics to draw
         ("entropy_given_prev", "Entropy given previous (bits)"), ("autocorr_lag1", "Autocorrelation (lag 1)"),
         ("entropy_value", "Entropy of the value (bits)"), ("max_l_per_min", "Largest reading (L/min)")]  # end of list
fig, axes = plt.subplots(2, 3, figsize=(13, 7))  # six small charts
for ax, (col, title) in zip(axes.flat, plots):  # one chart per metric
    ax.bar(fp.index, fp[col].astype(float))  # one bar per fixture
    ax.set_title(title)  # chart title
    ax.tick_params(axis="x", rotation=60)  # tilt the fixture names
fig.suptitle("Fingerprint per fixture (whole house excluded)")  # overall title
save_and_show("plot_fixture_fingerprint.png")  # fit, save and show the chart

### 10c. How is each fixture used?

The same event definition as in section 4, applied to every fixture. "Long" events are those of 60 s or more; this threshold was chosen for the shower and may not suit other fixtures.

In [ ]:
behaviour_rows = {}  # results per fixture
event_tables = {}  # keep the event tables
for name in fixtures:  # go through the fixtures
    ev_table = segment_events(frames[name])  # events of this fixture
    event_tables[name] = ev_table  # keep for later
    long_ev = ev_table[ev_table["event_class"] == "sustained"]  # long events only
    days_with_data = fp.loc[name, "days_with_data"]  # days with data of this fixture
    behaviour_rows[name] = {  # one row of the table
        "events": len(ev_table),  # all events
        "micro": int((ev_table["event_class"] == "micro").sum()),  # shorter than 10 s
        "medium": int((ev_table["event_class"] == "medium").sum()),  # 10 to 60 s
        "long": len(long_ev),  # 60 s or more
        "long_rows_share": float(long_ev["rows"].sum() / ev_table["rows"].sum()),  # share of flow readings in long events
        "events_per_day_with_data": float(len(ev_table) / days_with_data),  # all events per day with data
        "long_duration_median_s": float(long_ev["duration_s"].median()),  # typical duration of a long event
        "long_volume_median_l": float(long_ev["volume_l"].median()),  # typical approximate volume of a long event
        "long_mean_flow_median": float(long_ev["mean_flow"].median()),  # typical mean flow of a long event
    }  # end of row

behaviour = pd.DataFrame(behaviour_rows).T  # table: one row per fixture
display(behaviour.round(2))  # show the table

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))  # three charts side by side
axes[0].bar(behaviour.index, behaviour["events_per_day_with_data"].astype(float))  # activity level
axes[0].set_title("Events per day with data")  # chart title
axes[1].bar(behaviour.index, behaviour["long_volume_median_l"].astype(float))  # typical use
axes[1].set_title("Typical volume of a long event (L)")  # chart title
axes[2].bar(behaviour.index, behaviour["long_rows_share"].astype(float))  # where the flow readings are
axes[2].set_title("Share of flow readings in long events")  # chart title
for ax in axes:  # same tilt on all charts
    ax.tick_params(axis="x", rotation=60)  # tilt the fixture names
fig.suptitle("Use of each fixture")  # overall title
save_and_show("plot_fixture_behaviour.png")  # fit, save and show the chart

**Findings (measured, from the fingerprint of the earlier run).**
- Same in all fixtures: they are idle 81–99% of the time, and the entropy falls 47–75% once the previous value is known.
- Different: the washing machine is the hardest to predict. Only 12.8% of its rows are zero, 81.6% of its intervals are 2 s or less, and the entropy given the previous value is 2.60 bit, against 0.62 (bidet), 0.88 (shower), 1.27 (washbasin) and 1.44 (kitchen faucet). Its file has only 12,055 rows, so its entropy given the previous value may be biased low.
- The tables and plots in 10a and 10c (shared gaps, events per day, typical volumes) show the parts that were not part of the earlier run; read them before drawing conclusions.

**Limitation.** All fixtures come from one apartment and one measurement system; they are not independent sensors. The step grid was only tested for the shower, and the "long event" threshold was chosen for the shower.

## 11. Profile, summary and saved files

Everything above is collected in one dictionary (`profile`) and one table row (`fingerprint`), so the same checks can be repeated for another sensor and compared.

In [ ]:
profile["limitations"] = [  # what this analysis cannot tell
    "logging rule (heartbeat), counting window and pulse-to-value formula are inferred from the data, not documented",  # inference notice
    "sensor specification (2.25 ml per pulse, 1-30 L/min, +-10%) is the vendor specification of the sensor type, not verified for this unit",  # specification notice
    "duration of single-row events is unknown; event volumes are approximate",  # event limitation
    "cause of the 247-day gap and of the 2 ml/s micro events is unknown",  # gap limitation
    "one apartment and one measurement system; no other sensor type analysed",  # scope limitation
    "whole-house file not interpretable with the same format; excluded",  # data issue
]  # end of list
print(list(profile.keys()))  # check that all sections are present

In [ ]:
fingerprint = pd.DataFrame([{  # one row that describes this sensor
    "sensor": "shower_flow",  # sensor name
    "rows": profile["data_quality"]["row_count"],  # number of readings
    "days_with_data": profile["data_quality"]["days_with_data"],  # days that have data
    "zero_share_rows": profile["value_distribution"]["zero_fraction_by_rows"],  # zero share by rows
    "zero_share_time": profile["value_distribution"]["zero_fraction_by_time"],  # zero share by time
    "share_interval_le_2s": profile["data_quality"]["share_active_intervals"],  # share of intervals of 1-2 s
    "share_interval_300_302s": profile["data_quality"]["share_idle_intervals"],  # share of intervals of about 5 minutes
    "bits_pulse": profile["value_format"]["pulse_count_bits"],  # bits to hold a pulse count
    "entropy_pulse": profile["value_format"]["entropy_bits_pulse"],  # entropy of the values
    "entropy_pulse_diff": profile["value_format"]["entropy_bits_pulse_diff"],  # entropy of the changes
    "entropy_given_prev": profile["predictability"]["entropy_value_given_prev"],  # entropy given the previous value
    "autocorr_lag1": profile["predictability"]["autocorr_lag1"],  # correlation with the previous reading
    "events": profile["events"]["event_count"],  # number of events
    "sustained_rows_share": profile["events"]["class_summary"]["rows_share"]["sustained"],  # share of flow rows in long events
    "step_within_1_share": profile["noise"]["diff_within_1_share"],  # share of steps within one pulse
}])  # end of the row
fingerprint.T  # show it turned sideways for easy reading

In [ ]:
with open(OUT_DIR / "profile_shower.json", "w") as f:  # open the JSON file for writing
    json.dump(profile, f, indent=2)  # write the full profile

fingerprint.to_csv(OUT_DIR / "fingerprint_shower.csv", index=False)  # fingerprint of the shower
events.to_csv(OUT_DIR / "events_shower.csv")  # event table of the shower
fp.round(4).to_csv(OUT_DIR / "fingerprint_fixtures.csv")  # fixture statistics
behaviour.round(3).to_csv(OUT_DIR / "behaviour_fixtures.csv")  # fixture usage table
print(sorted(p.name for p in OUT_DIR.iterdir()))  # list the saved files

## Summary of findings

**Main sensor:** shower flow meter, 170,181 readings between 2019-02-13 and 2020-11-03. Only 358 of 629 calendar days contain data; the longest gap is 247 days. Four more fixtures are compared (washbasin, kitchen faucet, bidet, washing machine). The whole-house file could not be read with the same format and is excluded. Numbers come from a run on the dataset snapshot in the `WEUSEDTO-Data` repository; after re-running they should match the printed outputs.

**Measured findings, from simple to detailed**

1. **Not a regular series.** The logger writes every 1–2 s while water flows and about every 300 s while idle (45.4% and 53.3% of the intervals).
2. **Idle almost all the time.** The value is zero in 53.9% of rows but 98.75% of the time.
3. **Events.** 302 long (sustained) events hold 98.4% of all flowing readings; they start mostly at 07–09 h and 16–19 h (local time, assumed).
4. **Predictable from the past.** Lag-1 autocorrelation is 0.992; entropy falls from 3.50 bit (value alone) to 0.88 bit (given the previous value).
5. **Fixed step grid.** All 149 distinct values fit `value = floor(2.25 × k)` with k from 0 to 170, which agrees with the vendor specification of the sensor type (about 2.25 ml per pulse); the largest reading is about 23 L/min, inside the 1–30 L/min working range.
6. **Levels differ between events.** About half of the level variation is between events (23.6 ml/s) rather than within them (21.2 ml/s); 94.3% of the steps inside events are within ±1 pulse.
7. **Time follows the value.** After a zero reading the next gap is 300 s or more in 99.2% of the cases; after a non-zero reading it is shorter in 98.9%.
8. **Other fixtures follow the same pattern with different difficulty.** They are idle 81–99% of the time and the entropy falls 47–75% once the previous value is known; the washing machine is the hardest to predict.

**Additional checks in this notebook** (read the printed tables and plots): the sensor tail and working range (section 3), sizes of long events and recording quality inside events (section 4), drift by month (section 5), the constant check (section 7), shared gaps and use of each fixture (sections 10a and 10c).

**Interpretation (not proven):** the logger writes a heartbeat about every 5 minutes while idle; the sensor counts pulses; sustained events are showers. The dataset documentation does not confirm any of this.

## Assumptions and limitations

**Assumptions (not verified)**
- The value is flow in ml/s, as stated in the dataset publication.
- Local time is Europe/Rome (the apartment is assumed to be in Naples).
- Valid Unix times lie between 1.5e9 and 1.7e9 (mid-2017 to late 2023); this removes one broken Washbasin row.
- The zero share by time assumes each value holds until the next reading, and ignores gaps over one hour.
- Event definition: value > 0, split at gaps over 60 s; volume = flow × time to the next reading, capped at 5 s.
- The sensor specification (2.25 ml per pulse, 1–30 L/min, ±10%) is the vendor specification of the YF-S201 type; it is assumed to apply to the sensors in this data set.

**Inferred from the data, not documented**
- The logger writes a heartbeat about every 300 s while idle.
- The values lie on the grid `floor(2.25 × k)`; the counting window and the formula are unverified.
- Sustained events are showers; micro events are drips, residual water or readings below the specified range.

**Limitations**
- One apartment, one measurement system; no vibration or other sensor type.
- Single-row events have unknown duration; the cause of the 247-day gap is unknown.
- Entropy estimates are in-sample.
- The whole-house file could not be interpreted and was excluded; simultaneous use of several fixtures was not analysed.